# Early-season prediction audit

Analysis-only evaluation of stored leakage-safe outer-fold predictions. No fitting, retraining, feature changes, or optimizer changes are performed.

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import duckdb

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB = ROOT / 'data/db/euroleague.duckdb'
MINUTES_PATH = ROOT / 'data/derived/phase4b/components/p4b__minutes__catboost__role_older_history/outer_predictions.parquet'
FP_PATH = ROOT / 'data/derived/phase6c/research/outer_predictions.parquet'
FEATURE_PATH = ROOT / 'data/derived/phase6c/research/feature_frame.parquet'

con = duckdb.connect()
minutes_raw = con.execute('SELECT * FROM read_parquet(?)', [str(MINUTES_PATH)]).df()
fp_raw = con.execute('SELECT * FROM read_parquet(?)', [str(FP_PATH)]).df()
features_raw = con.execute('SELECT * FROM read_parquet(?)', [str(FEATURE_PATH)]).df()
print('minutes', minutes_raw.shape, minutes_raw.columns.tolist())
print('fp', fp_raw.shape, fp_raw.columns.tolist())
print('features', features_raw.shape)
print(features_raw.columns.tolist())

minutes (22399, 25) ['model_row_id', 'season', 'round_number', 'game_id', 'player_id', 'team_id', 'opponent_team_id', 'target_game_time', 'fantasy_matchday', 'fantasy_position', 'target_minutes', 'actual_fantasy_points', 'career_el_games_before', 'competition_stage', 'postseason_game', 'role_change_score', 'experiment_id', 'model_version', 'outer_fold', 'prediction_role', 'model_type', 'feature_manifest', 'target_strategy', 'predicted_minutes', 'actual_minutes']
fp (22399, 95) ['model_row_id', 'season', 'round_number', 'game_id', 'player_id', 'team_id', 'opponent_team_id', 'target_game_time', 'feature_cutoff_time', 'fantasy_matchday', 'fantasy_position', 'target_minutes', 'career_el_games_before', 'season_games_before', 'season_minutes_avg_before', 'season_fp_std_before', 'last_5_fp_std', 'last_5_start_rate', 'rot_last5_starter_rate', 'rot_last5_closing_lineup_rate', 'outer_fold', 'actual_fp', 'phase4b_central_fp', 'mean_challenger_fp', 'robust_mean_challenger_fp', 'raw_q10', 'cal_q10'

In [4]:
# Assemble the held-out audit frame and prior-season-only context.
fp_cols = ['model_row_id','season','round_number','game_id','player_id','team_id','opponent_team_id','target_game_time','outer_fold','career_el_games_before','season_games_before','actual_fp','phase6c_expected_fp']
min_cols = ['model_row_id','predicted_minutes','actual_minutes','experiment_id','target_strategy']
feature_cols = ['model_row_id','season_start_year','last_5_minutes_avg','minutes_ewma','season_minutes_avg_before']
audit = fp_raw[fp_cols].merge(minutes_raw[min_cols], on='model_row_id', validate='one_to_one').merge(features_raw[feature_cols], on='model_row_id', validate='one_to_one')
audit = audit.rename(columns={'phase6c_expected_fp':'predicted_fp'})
audit['matchday'] = audit['round_number'].astype(int)
audit['minutes_error'] = audit['predicted_minutes'] - audit['actual_minutes']
audit['fp_error'] = audit['predicted_fp'] - audit['actual_fp']

history = features_raw[['season_start_year','season','player_id','team_id','target_game_time','target_minutes']].copy()
history = history[pd.to_numeric(history.target_minutes, errors='coerce').gt(0)].sort_values(['player_id','season_start_year','target_game_time'])
prior = history.groupby(['player_id','season_start_year'], as_index=False).agg(prev_season_games=('target_minutes','size'), prev_season_mpg=('target_minutes','mean'), prev_team_id=('team_id','last'))
prior_last5 = history.groupby(['player_id','season_start_year']).tail(5).groupby(['player_id','season_start_year'], as_index=False).agg(prev_season_last5_mpg=('target_minutes','mean'))
prior = prior.merge(prior_last5, on=['player_id','season_start_year'], validate='one_to_one').rename(columns={'season_start_year':'prior_season_start_year'})
audit['prior_season_start_year'] = audit['season_start_year'] - 1
audit = audit.merge(prior, on=['player_id','prior_season_start_year'], how='left', validate='many_to_one')
audit['returning_player'] = audit['prev_season_games'].notna()
audit['changed_team'] = audit['returning_player'] & audit['team_id'].ne(audit['prev_team_id'])
audit['new_el_player'] = audit['career_el_games_before'].eq(0)

db_con = duckdb.connect(str(DB), read_only=True)
names = db_con.execute('SELECT canonical_player_id AS player_id, canonical_name AS player FROM players').df()
db_con.close()
audit = audit.merge(names, on='player_id', how='left', validate='many_to_one')
audit['player'] = audit['player'].fillna(audit['player_id'])

def role_bucket(value):
    if pd.isna(value): return 'No previous season'
    if value >= 28: return 'Star / heavy role (28+)'
    if value >= 22: return 'Starter (22–<28)'
    if value >= 14: return 'Rotation (14–<22)'
    return 'Bench (<14)'

audit['previous_role'] = audit['prev_season_mpg'].map(role_bucket)

assert len(audit) == len(fp_raw) == len(minutes_raw)
assert audit.model_row_id.is_unique
assert set(audit.season) == {'E2023','E2024','E2025'}
assert (audit.actual_minutes > 0).all()
assert audit[['predicted_minutes','actual_minutes','predicted_fp','actual_fp']].notna().all().all()
print(audit.groupby('season').agg(rows=('model_row_id','size'), min_round=('matchday','min'), max_round=('matchday','max'), fantasy_md_present=('model_row_id', lambda x: fp_raw.loc[x.index, 'fantasy_matchday'].notna().sum())))
print('early <=5 rows', len(audit[audit.matchday <= 5]), 'returning MD1', len(audit[(audit.matchday == 1) & audit.returning_player]))
print('minutes artifact', audit.experiment_id.unique().tolist())

        rows  min_round  max_round  fantasy_md_present
season                                                
E2023   6984          1         43                5797
E2024   6899          1         43                5944
E2025   8516          1         47                7735
early <=5 rows 2772 returning MD1 419
minutes artifact ['p4b__minutes__catboost__role_older_history']


In [5]:
def safe_corr(a, b, method='pearson'):
    pair = pd.DataFrame({'a':pd.to_numeric(a, errors='coerce'),'b':pd.to_numeric(b, errors='coerce')}).dropna()
    if len(pair) < 2 or pair.a.nunique() < 2 or pair.b.nunique() < 2: return np.nan
    if method == 'spearman': return pair.a.rank(method='average').corr(pair.b.rank(method='average'))
    return pair.a.corr(pair.b)

def metrics(frame, pred, actual, tolerance=True):
    values = frame[[pred, actual]].dropna().astype(float)
    err = values[pred] - values[actual]
    out = {'N':len(values), 'MAE':err.abs().mean(), 'RMSE':np.sqrt(np.mean(err**2)), 'Bias (pred−actual)':err.mean(), 'Median error':err.median()}
    if tolerance:
        out.update({'Within ±3':(err.abs() <= 3).mean(), 'Within ±5':(err.abs() <= 5).mean(), '|error| >10':(err.abs() > 10).mean()})
    out.update({'Pearson':safe_corr(values[pred], values[actual]), 'Spearman':safe_corr(values[pred], values[actual], 'spearman')})
    return out

window_masks = {'MD1':audit.matchday.eq(1), 'MD1–3':audit.matchday.le(3), 'MD1–5':audit.matchday.le(5)}
minute_rows, fp_rows = [], []
for season in [*sorted(audit.season.unique()), 'Pooled']:
    season_mask = pd.Series(True, index=audit.index) if season == 'Pooled' else audit.season.eq(season)
    for window, wmask in window_masks.items():
        sample = audit[season_mask & wmask]
        minute_rows.append({'Season':season, 'Window':window, **metrics(sample,'predicted_minutes','actual_minutes')})
        fp_rows.append({'Season':season, 'Window':window, **metrics(sample,'predicted_fp','actual_fp', tolerance=False)})
minutes_accuracy = pd.DataFrame(minute_rows)
fp_accuracy = pd.DataFrame(fp_rows)

returning = audit[audit.returning_player].copy()
baseline_rows = []
for window, wmask in window_masks.items():
    sample = returning[wmask.loc[returning.index]]
    for label, column in [('Frozen minutes model','predicted_minutes'),('Previous-season MPG','prev_season_mpg'),('Previous-season last5 MPG','prev_season_last5_mpg'),('Existing pregame minutes EWMA','minutes_ewma')]:
        baseline_rows.append({'Window':window,'Predictor':label, **metrics(sample,column,'actual_minutes')})
baseline_accuracy = pd.DataFrame(baseline_rows)

md1_returning = audit[audit.matchday.eq(1) & audit.returning_player].copy()
role_rows = []
role_order = ['Star / heavy role (28+)','Starter (22–<28)','Rotation (14–<22)','Bench (<14)']
for role in role_order:
    sample = md1_returning[md1_returning.previous_role.eq(role)]
    role_rows.append({'Previous-season role':role,'N':len(sample),'Previous MPG':sample.prev_season_mpg.mean(),'Predicted MD1':sample.predicted_minutes.mean(),'Actual MD1':sample.actual_minutes.mean(),'Pred−previous':(sample.predicted_minutes-sample.prev_season_mpg).mean(),'Bias (pred−actual)':sample.minutes_error.mean(),'MAE':sample.minutes_error.abs().mean()})
role_accuracy = pd.DataFrame(role_rows)

pred_slope = np.polyfit(md1_returning.prev_season_mpg, md1_returning.predicted_minutes, 1)[0]
actual_slope = np.polyfit(md1_returning.prev_season_mpg, md1_returning.actual_minutes, 1)[0]
compression = {'previous_mpg_sd':md1_returning.prev_season_mpg.std(),'predicted_md1_sd':md1_returning.predicted_minutes.std(),'actual_md1_sd':md1_returning.actual_minutes.std(),'pred_on_previous_slope':pred_slope,'actual_on_previous_slope':actual_slope,'pred_previous_corr':safe_corr(md1_returning.prev_season_mpg,md1_returning.predicted_minutes),'actual_previous_corr':safe_corr(md1_returning.prev_season_mpg,md1_returning.actual_minutes)}

history_labels = pd.cut(audit.season_games_before, bins=[-1,0,2,5,np.inf], labels=['0 games','1–2 games','3–5 games','6+ games'])
cold_rows = []
for label in history_labels.cat.categories:
    sample = audit[history_labels.eq(label)]
    cold_rows.append({'Current-season history':label,'N':len(sample),'Minutes MAE':sample.minutes_error.abs().mean(),'Minutes bias':sample.minutes_error.mean(),'FP MAE':sample.fp_error.abs().mean(),'FP bias':sample.fp_error.mean()})
cold_start = pd.DataFrame(cold_rows)
cumulative_rows = []
for label, mask in [('0 games',audit.season_games_before.eq(0)),('≤2 games',audit.season_games_before.le(2)),('≤5 games',audit.season_games_before.le(5))]:
    sample=audit[mask]
    cumulative_rows.append({'History threshold':label,'N':len(sample),'Minutes MAE':sample.minutes_error.abs().mean(),'FP MAE':sample.fp_error.abs().mean()})
cold_cumulative=pd.DataFrame(cumulative_rows)

early5 = audit[audit.matchday.le(5)].copy()
error_link = {'N':len(early5),'Pearson errors':safe_corr(early5.minutes_error,early5.fp_error),'Spearman errors':safe_corr(early5.minutes_error,early5.fp_error,'spearman')}
error_link['Simple R²'] = error_link['Pearson errors'] ** 2
error_link['FP-error slope per minute'] = np.polyfit(early5.minutes_error, early5.fp_error, 1)[0]
error_link['FP MAE when |minutes error|≤3'] = early5.loc[early5.minutes_error.abs().le(3),'fp_error'].abs().mean()
error_link['FP MAE when |minutes error|>10'] = early5.loc[early5.minutes_error.abs().gt(10),'fp_error'].abs().mean()

subset_defs = {
    'High-minute stars (prior MPG 28+)':early5.returning_player & early5.prev_season_mpg.ge(28),
    'New EuroLeague players':early5.new_el_player,
    'Returning, changed team':early5.returning_player & early5.changed_team,
    'Returning, same team':early5.returning_player & ~early5.changed_team,
}
subset_rows=[]
for label, mask in subset_defs.items():
    sample=early5[mask]
    subset_rows.append({'Group':label,'N':len(sample),'Minutes MAE':sample.minutes_error.abs().mean(),'Minutes bias':sample.minutes_error.mean(),'FP MAE':sample.fp_error.abs().mean(),'FP bias':sample.fp_error.mean()})
subset_accuracy=pd.DataFrame(subset_rows)

miss_cols=['player','season','matchday','prev_season_mpg','predicted_minutes','actual_minutes','predicted_fp','actual_fp','career_el_games_before']
minutes_under=early5.assign(miss=early5.actual_minutes-early5.predicted_minutes).nlargest(15,'miss')[miss_cols+['miss']]
minutes_over=early5.assign(miss=early5.predicted_minutes-early5.actual_minutes).nlargest(15,'miss')[miss_cols+['miss']]
fp_under=early5.assign(miss=early5.actual_fp-early5.predicted_fp).nlargest(15,'miss')[miss_cols+['miss']]
fp_over=early5.assign(miss=early5.predicted_fp-early5.actual_fp).nlargest(15,'miss')[miss_cols+['miss']]

STRATEGY_PATH=ROOT/'data/derived/phase7/research/e2025_strategy_backtest.parquet'
strategy=con.execute('SELECT * FROM read_parquet(?)',[str(STRATEGY_PATH)]).df()
budget_sample=strategy[strategy.strategy.eq('DETERMINISTIC_EXPECTED') & strategy.matchday.le(5)][['season','matchday','strategy','available_budget','total_credits','bank_after']]

print(minutes_accuracy.to_string(index=False))
print()
print('ROLE')
print(role_accuracy.to_string(index=False))
print()
print('BASELINES')
print(baseline_accuracy[['Window','Predictor','N','MAE','Bias (pred−actual)']].to_string(index=False))
print()
print('COLD')
print(cold_start.to_string(index=False))
print()
print('ERROR LINK', error_link)
print()
print('SUBSETS')
print(subset_accuracy.to_string(index=False))
print()
print('COMPRESSION', compression)
print()
print('BUDGET')
print(budget_sample.to_string(index=False))
print('average unused', budget_sample.bank_after.mean())

Season Window    N      MAE     RMSE  Bias (pred−actual)  Median error  Within ±3  Within ±5  |error| >10  Pearson  Spearman
 E2023    MD1  155 6.410303 7.903548           -1.738149     -1.745586   0.290323   0.438710     0.232258 0.555629  0.541980
 E2023  MD1–3  530 5.144443 6.463030           -0.693699     -0.943235   0.366038   0.556604     0.118868 0.636642  0.635838
 E2023  MD1–5  910 4.979303 6.305351           -0.649000     -0.871569   0.376923   0.583516     0.108791 0.645022  0.641041
 E2024    MD1  139 5.076162 6.561920           -0.417552     -1.158614   0.402878   0.611511     0.115108 0.507508  0.512630
 E2024  MD1–3  506 4.731334 6.079692            0.226417      0.132906   0.416996   0.614625     0.094862 0.623209  0.623442
 E2024  MD1–5  873 4.501210 5.797781           -0.092516     -0.195173   0.436426   0.631157     0.087056 0.660643  0.667336
 E2025    MD1  151 5.381942 6.708759            0.292233     -0.390520   0.324503   0.549669     0.125828 0.473179  0.463926


In [ ]:
def markdown_table(frame, columns, labels=None, digits=1):
    labels = labels or columns
    lines = ['| ' + ' | '.join(labels) + ' |', '| ' + ' | '.join(['---'] + ['---:'] * (len(columns)-1)) + ' |']
    for _, row in frame[columns].iterrows():
        values=[]
        for value in row:
            if pd.isna(value): values.append('—')
            elif isinstance(value, (float, np.floating)): values.append(f'{value:.{digits}f}')
            else: values.append(str(value).replace('|','/'))
        lines.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join(lines)

miss_labels=['Player','Season','MD','Prev MPG','Pred MIN','Actual MIN','Pred FP','Actual FP','Prior EL games','Miss']
for title, frame in [('15 largest minutes underpredictions',minutes_under),('15 largest minutes overpredictions',minutes_over),('15 largest FP underpredictions',fp_under),('15 largest FP overpredictions',fp_over)]:
    print('### '+title)
    print(markdown_table(frame, miss_cols+['miss'], miss_labels, 1))
    print()